# Frozen network-candidate union checkpoint

This notebook reads **compact tracked products only**. It does not open DAS
waveforms, network miniSEED, full score arrays, or any held-out interval.

The development result is deliberately mixed: the template bank passed its
registered target-injection gate, while the generic trigger failed at component
SNR 1.0. The generic branch is retained without retuning as an auxiliary
non-template safety net. The network union is now frozen for the next
**independent DAS-only development** stage, not for held-out evaluation.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "outputs" / "development_network").is_dir():
    candidates = [ROOT, *ROOT.parents]
    matches = [
        path for path in candidates
        if (path / "outputs" / "development_network").is_dir()
    ]
    if not matches:
        raise FileNotFoundError("Run from repeaters_v2 or one of its subdirectories")
    ROOT = matches[0]

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DEVELOPMENT = ROOT / "outputs" / "development_network"
CONFIG = ROOT / "config"
pd.set_option("display.max_columns", 40)
print("Project:", ROOT)
print("Inputs: compact CSV/JSON only")

## 1. What is frozen—and what is not

CONDITIONAL is intentional. It preserves the generic branch's failed SNR-1
gate while allowing it to remain in a conservative network union. No family
membership is assigned here, and no candidate is called new merely because it
is absent from the local routine catalog.

In [ ]:
with (DEVELOPMENT / "network_union_status.json").open(encoding="utf-8") as handle:
    status = json.load(handle)
with (CONFIG / "network_union.json").open(encoding="utf-8") as handle:
    union_config = json.load(handle)

gate_fields = [
    "status",
    "freeze_status",
    "network_detector_performance_status",
    "template_branch_role",
    "generic_branch_role",
    "target_snr1_template_recovery",
    "target_snr1_generic_recovery",
    "time_only_union_candidate_count",
    "known_target_region_event_count",
    "known_regional_arrival_veto_count",
    "unassociated_after_broader_catalog_count",
    "network_union_stage_das_waveforms_opened",
    "heldout_intervals_opened",
    "next_stage_gate",
    "heldout_access_gate",
]
display(pd.Series({field: status[field] for field in gate_fields}, name="value"))

## 2. Time-only union before catalog adjudication

The table below contains no catalog event IDs or family labels. Cross-branch
matching uses only ordered candidate times and branch identity. Template times
are estimated origins; generic times are arrivals. When both branches match,
the template origin is the representative time.

In [ ]:
template = pd.read_csv(DEVELOPMENT / "candidate_detections.csv")
generic = pd.read_csv(DEVELOPMENT / "generic_candidate_detections.csv")
time_only = pd.read_csv(
    DEVELOPMENT / "network_candidate_union_time_only.csv",
    dtype={"template_candidate_id": str, "generic_candidate_id": str},
)

start = pd.Timestamp("2025-01-20T04:55:00Z").timestamp()
template_x = template["origin_epoch_s"] - start
generic_x = generic["trigger_epoch_s"] - start

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.scatter(template_x, [1] * len(template), s=75, label="template origin", zorder=3)
ax.scatter(generic_x, [0] * len(generic), s=75, label="generic arrival", zorder=3)
for row in time_only.itertuples():
    if pd.notna(row.template_origin_epoch_s) and pd.notna(row.generic_trigger_epoch_s):
        ax.plot(
            [row.template_origin_epoch_s - start, row.generic_trigger_epoch_s - start],
            [1, 0],
            color="0.55",
            linewidth=1.5,
            zorder=1,
        )
ax.set(
    xlabel="seconds after 2025-01-20 04:55:00 UTC",
    yticks=[0, 1],
    yticklabels=["generic", "template"],
    title="Frozen time-only cross-branch union",
)
ax.grid(axis="x", alpha=0.25)
ax.legend(loc="upper right")
plt.show()

display(time_only[[
    "union_candidate_id",
    "representative_time",
    "branch_membership",
    "template_candidate_id",
    "generic_candidate_id",
    "cross_branch_time_difference_s",
    "catalog_fields_used_in_grouping",
    "family_assignment",
]])

## 3. Advisor sandbox: matching-window sensitivity

Change EXPLORATORY_MATCH_WINDOW_S and rerun this cell. This recomputes an
in-memory table only. It does **not** overwrite the registered 8-second union,
change either detector threshold, or authorize DAS/held-out access.

The registered 8 seconds reuses both v4 branches' already declared minimum
candidate separation; it was not tuned after seeing DAS.

In [ ]:
from src.network_union import build_time_only_union

EXPLORATORY_MATCH_WINDOW_S = 8.0

sandbox_rows = build_time_only_union(
    template.to_dict("records"),
    generic.to_dict("records"),
    maximum_difference_s=EXPLORATORY_MATCH_WINDOW_S,
    identifier_prefix="sandbox_union",
)
sandbox = pd.DataFrame(sandbox_rows)
print("Exploratory window (s):", EXPLORATORY_MATCH_WINDOW_S)
print("Frozen window (s):", status["cross_branch_match_window_s"])
print("Exploratory event groups:", len(sandbox))
print("Frozen event groups:", status["time_only_union_candidate_count"])
display(sandbox[[
    "union_candidate_id",
    "representative_time",
    "branch_membership",
    "cross_branch_time_difference_s",
]])

## 4. Catalog evidence is attached afterward

The raw union keeps all three groups. The broader catalog audit then identifies
the generic-only group as a physically plausible arrival from NC event
75120096 near Carpinteria, about 197 km away. That row remains auditable but is
excluded from an *uncataloged local extension* count. The two joint groups are
known local events. None receives a repeater-family assignment.

In [ ]:
adjudicated = pd.read_csv(
    DEVELOPMENT / "network_candidate_union_adjudicated.csv",
    dtype={
        "target_region_catalog_event_id": str,
        "broader_catalog_event_id": str,
    },
)
display(adjudicated[[
    "union_candidate_id",
    "branch_membership",
    "target_region_catalog_event_id",
    "broader_catalog_event_id",
    "broader_catalog_location_name",
    "broader_catalog_horizontal_distance_km",
    "known_event_class",
    "local_extension_disposition",
    "eligible_uncataloged_local_extension_candidate",
]])

## Checkpoint decision

- The network candidate union is frozen for **independent DAS-only development**.
- It is not declared a fully passing detector: the generic SNR-1 STOP remains.
- In this development interval, the network union has two known local events,
  one known regional arrival, and zero unassociated local candidates.
- The next detector must generate DAS candidates without importing these
  network times or catalog event times.
- The 12 held-out hours remain sealed.

This is the comparison that makes a later DAS-extension claim credible: DAS
must add independently adjudicated events beyond this full network union, not
merely beyond the routine seismic catalog.